<a href="https://colab.research.google.com/github/lilian662/PROCESOS-ESTOCASTICOS/blob/main/Ejercicios_programados_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#EJERCICIO PROGRAMADOS
Edith Lilian Castillo Ojeda

# Ejercicio 3. Aproximación de la matriz de transición \(P(t)\)

Para una Cadena de Markov en Tiempo Continuo (CMTC), la matriz de probabilidades de transición \(P(t)\) describe la probabilidad de pasar de un estado \(i\) a un estado \(j\) después de un tiempo \(t\).

Utilizando el teorema de uniformización, la matriz \(P(t)\) puede expresarse como:

$$
P(t)=\sum_{k=0}^{\infty} e^{-rt}\frac{(rt)^k}{k!}\hat{P}^{k}
$$

donde:

- \(r\) es una constante tal que \(r \geq \max_i r_i\).
- \(\hat{P}\) es la matriz de transición de la cadena embebida.
- \(\hat{P}^{k}\) representa la matriz de transición después de \(k\) pasos.
-

$$
e^{-rt}\frac{(rt)^k}{k!}
$$

corresponde a la probabilidad de que un proceso de Poisson de tasa \(r\) tenga exactamente \(k\) eventos en el intervalo \([0,t]\).

Como la serie es infinita, se aproxima utilizando los primeros \(M\) términos:



$$
P(t)\approx \sum_{k=0}^{M} e^{-rt}\frac{(rt)^k}{k!}\hat P^{k}
$$

De acuerdo con la recomendación del teorema, se elige:

$$
M \approx \max\left\{rt+5\sqrt{rt},\,20\right\}
$$

lo que proporciona una aproximación suficientemente precisa para la matriz de transición.



In [1]:
import numpy as np
import math

R = np.array([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
], dtype=float)

r = 6

ri = R.sum(axis=1)

P_hat = R / r

for i in range(4):
    P_hat[i, i] = 1 - ri[i] / r

print("Matriz P_hat:")
print(P_hat)

Matriz P_hat:
[[0.16666667 0.33333333 0.5        0.        ]
 [0.66666667 0.         0.33333333 0.        ]
 [0.         0.33333333 0.33333333 0.33333333]
 [0.16666667 0.         0.5        0.33333333]]


In [2]:
def calcular_P(t):
    M = max(math.ceil(r*t + 5*math.sqrt(r*t)), 20)

    P = np.zeros((4, 4))
    potencia = np.eye(4)

    for k in range(M + 1):
        coef = math.exp(-r*t) * (r*t)**k / math.factorial(k)
        P = P + coef * potencia
        potencia = potencia @ P_hat

    return M, P

In [3]:
for t in [0.5, 1, 5]:
    M, P = calcular_P(t)
    print(f"\nP({t}) con M = {M}")
    print(np.round(P, 6))


P(0.5) con M = 20
[[0.250609 0.216965 0.386657 0.14577 ]
 [0.253135 0.238361 0.374409 0.134095]
 [0.169119 0.193615 0.420301 0.216965]
 [0.158017 0.157445 0.398332 0.286206]]

P(1) con M = 20
[[0.206151 0.203902 0.39871  0.191236]
 [0.208284 0.205341 0.397899 0.188474]
 [0.196758 0.198379 0.400959 0.203902]
 [0.192046 0.193997 0.401471 0.212484]]

P(5) con M = 58
[[0.2      0.2      0.399999 0.2     ]
 [0.2      0.2      0.399999 0.2     ]
 [0.2      0.2      0.399999 0.2     ]
 [0.2      0.2      0.399999 0.2     ]]


In [4]:
M05, P05 = calcular_P(0.5)
M1, P1 = calcular_P(1)

producto = P05 @ P05

print("P(1):")
print(np.round(P1, 6))

print("\nP(0.5)P(0.5):")
print(np.round(producto, 6))

print("\nDiferencia:")
print(np.round(P1 - producto, 8))

P(1):
[[0.206151 0.203902 0.39871  0.191236]
 [0.208284 0.205341 0.397899 0.188474]
 [0.196758 0.198379 0.400959 0.203902]
 [0.192046 0.193997 0.401471 0.212484]]

P(0.5)P(0.5):
[[0.206151 0.203902 0.39871  0.191236]
 [0.208285 0.205341 0.3979   0.188475]
 [0.196759 0.19838  0.400959 0.203902]
 [0.192047 0.193997 0.401472 0.212485]]

Diferencia:
[[-2.9e-07 -2.9e-07 -5.8e-07 -2.9e-07]
 [-2.9e-07 -2.9e-07 -5.8e-07 -2.9e-07]
 [-2.9e-07 -2.9e-07 -5.8e-07 -2.9e-07]
 [-2.9e-07 -2.9e-07 -5.8e-07 -2.9e-07]]


## Conclusión

Se calculó la matriz de transición \(P(t)\) para distintos valores de tiempo utilizando el método de uniformización. Los resultados muestran cómo evolucionan las probabilidades de transición de la cadena conforme transcurre el tiempo.



# Ejercicio 4. Cotas de error para \(P(t)\)


La matriz de transición de una Cadena de Markov en Tiempo Continuo puede calcularse mediante la serie de uniformización:

$$
P(t)=\sum_{k=0}^{\infty} e^{-rt}\frac{(rt)^k}{k!}\hat{P}^{k}
$$

Sin embargo, debido a que la serie contiene infinitos términos, en la práctica se utiliza una aproximación truncada:

$$
P^{M}(t)=\sum_{k=0}^{M} e^{-rt}\frac{(rt)^k}{k!}\hat{P}^{k}
$$

El teorema de cotas de error establece que:

$$
\left|p_{ij}(t)-p_{ij}^{M}(t)\right|
\le
\sum_{k=M+1}^{\infty}
e^{-rt}\frac{(rt)^k}{k!}
$$

para todo

$$
1 \le i,j \le N.
$$

Esta desigualdad permite controlar el error cometido al truncar la serie y elegir un valor adecuado de \(M\).


In [5]:
import numpy as np
import math

R = np.array([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
], dtype=float)

r = 6
epsilon = 0.00001

ri = R.sum(axis=1)

P_hat = R / r

for i in range(4):
    P_hat[i, i] = 1 - ri[i] / r

print("Matriz P_hat:")
print(P_hat)

Matriz P_hat:
[[0.16666667 0.33333333 0.5        0.        ]
 [0.66666667 0.         0.33333333 0.        ]
 [0.         0.33333333 0.33333333 0.33333333]
 [0.16666667 0.         0.5        0.33333333]]


In [9]:
def uniformizacion(t, epsilon):
    A = np.eye(4)
    B = math.exp(-r*t) * A

    c = math.exp(-r*t)
    suma = c
    k = 1

    while suma < 1 - epsilon:
        c = c * (r*t) / k
        A = A @ P_hat
        B = B + c * A
        suma = suma + c
        k = k + 1

    M = k - 1
    return M, B

In [10]:
for t in [0.5, 1, 5]:
    M, P = uniformizacion(t, epsilon)

    print(f"\nP({t}) con tolerancia epsilon = {epsilon}")
    print(f"Valor de M = {M}")
    print(np.round(P, 6))


P(0.5) con tolerancia epsilon = 1e-05
Valor de M = 13
[[0.250608 0.216964 0.386656 0.145769]
 [0.253134 0.23836  0.374408 0.134094]
 [0.169119 0.193614 0.4203   0.216964]
 [0.158017 0.157444 0.39833  0.286205]]

P(1) con tolerancia epsilon = 1e-05
Valor de M = 19
[[0.20615  0.203901 0.398708 0.191235]
 [0.208283 0.20534  0.397898 0.188474]
 [0.196758 0.198379 0.400957 0.203901]
 [0.192045 0.193996 0.401469 0.212483]]

P(5) con tolerancia epsilon = 1e-05
Valor de M = 56
[[0.199999 0.199999 0.399997 0.199999]
 [0.199999 0.199999 0.399997 0.199999]
 [0.199999 0.199999 0.399997 0.199999]
 [0.199999 0.199999 0.399997 0.199999]]


Para comparar el ejercicio 3

In [6]:
def aproximacion_ejercicio3(t):
    M = max(math.ceil(r*t + 5*math.sqrt(r*t)), 20)

    P = np.zeros((4, 4))
    A = np.eye(4)

    for k in range(M + 1):
        c = math.exp(-r*t) * (r*t)**k / math.factorial(k)
        P = P + c * A
        A = A @ P_hat

    return M, P

In [11]:
for t in [0.5, 1, 5]:
    M3, P3 = aproximacion_ejercicio3(t)
    M4, P4 = uniformizacion(t, epsilon)

    print(f"\nComparación para t = {t}")
    print(f"M ejercicio 3 = {M3}")
    print(f"M ejercicio 4 = {M4}")

    print("\nDiferencia P3 - P4:")
    print(np.round(P3 - P4, 8))


Comparación para t = 0.5
M ejercicio 3 = 20
M ejercicio 4 = 13

Diferencia P3 - P4:
[[6.80e-07 6.80e-07 1.36e-06 6.80e-07]
 [6.80e-07 6.80e-07 1.36e-06 6.80e-07]
 [6.80e-07 6.80e-07 1.36e-06 6.80e-07]
 [6.80e-07 6.80e-07 1.36e-06 6.80e-07]]

Comparación para t = 1
M ejercicio 3 = 20
M ejercicio 4 = 19

Diferencia P3 - P4:
[[7.50e-07 7.50e-07 1.49e-06 7.50e-07]
 [7.50e-07 7.50e-07 1.49e-06 7.50e-07]
 [7.50e-07 7.50e-07 1.49e-06 7.50e-07]
 [7.50e-07 7.50e-07 1.49e-06 7.50e-07]]

Comparación para t = 5
M ejercicio 3 = 58
M ejercicio 4 = 56

Diferencia P3 - P4:
[[1.1e-06 1.1e-06 2.2e-06 1.1e-06]
 [1.1e-06 1.1e-06 2.2e-06 1.1e-06]
 [1.1e-06 1.1e-06 2.2e-06 1.1e-06]
 [1.1e-06 1.1e-06 2.2e-06 1.1e-06]]


Y para verificar otra vez Chapman-Kolmogorov:

In [12]:
M05, P05 = uniformizacion(0.5, epsilon)
M1, P1 = uniformizacion(1, epsilon)

producto = P05 @ P05

print("P(1):")
print(np.round(P1, 6))

print("\nP(0.5)P(0.5):")
print(np.round(producto, 6))

print("\nDiferencia:")
print(np.round(P1 - producto, 8))

P(1):
[[0.20615  0.203901 0.398708 0.191235]
 [0.208283 0.20534  0.397898 0.188474]
 [0.196758 0.198379 0.400957 0.203901]
 [0.192045 0.193996 0.401469 0.212483]]

P(0.5)P(0.5):
[[0.20615  0.203901 0.398707 0.191235]
 [0.208283 0.20534  0.397897 0.188473]
 [0.196757 0.198378 0.400957 0.203901]
 [0.192045 0.193996 0.401469 0.212483]]

Diferencia:
[[3.2e-07 3.2e-07 6.5e-07 3.2e-07]
 [3.2e-07 3.2e-07 6.5e-07 3.2e-07]
 [3.2e-07 3.2e-07 6.5e-07 3.2e-07]
 [3.2e-07 3.2e-07 6.5e-07 3.2e-07]]


##Conclusión

Se implementó el algoritmo de uniformización utilizando una tolerancia de

$$
\varepsilon = 0.00001
$$

El método permitió calcular las matrices de transición \(P(0.5)\), \(P(1)\) y \(P(5)\) asegurando que el error de aproximación fuera menor que la tolerancia establecida.

Al comparar los resultados con los obtenidos en el ejercicio 3, se observa que ambas aproximaciones son prácticamente iguales, lo que confirma la validez del método y de la cota de error utilizada.

Además, el algoritmo ajusta automáticamente el valor de \(M\), evitando realizar cálculos innecesarios y garantizando al mismo tiempo la precisión requerida.